In [9]:
import vaex

# Load your Parquet file
df = vaex.open("D:/jn/Amazon_cs_proj/Issue_categor_proj_Phase1/final_phase_ready_vaex/*.parquet")

# Define your label columns (multi-label binary columns)
issue_columns = [
    'refund_and_returns',
    'delivery_delay',
    'product_quality',
    'order_issue',
    'packaging_handling',
    'long_wait_times',
    'technical_support',
    'warranty_guarantee',
    'fake_counterfeit',
    'rare_issues',
    'minor_issues'
]

text_column = 'combined_review'  # Replace with the actual name of your text column

# Output FastText training file
output_file = "fasttext_training_data.txt"

# Choose a reasonable chunk size (like 100_000 or 1_000_000)
chunk_size = 1_000_000
total_rows = len(df)

with open(output_file, "w", encoding="utf-8") as f:
    for start in range(0, total_rows, chunk_size):
        end = min(start + chunk_size, total_rows)
        chunk = df[start:end]

        # Convert text column and label columns to lists
        texts = chunk[text_column].tolist()
        all_labels = [chunk[col].tolist() for col in issue_columns]

        for i in range(len(texts)):
            try:
                text = texts[i]
                if not isinstance(text, str) or not text.strip():
                    continue  # Skip blank or non-string text

                # Get active labels
                labels = [
                    f"__label__{issue_columns[j]}"
                    for j in range(len(issue_columns))
                    if all_labels[j][i] == 1
                ]

                if labels:
                    clean_text = text.replace("\n", " ").strip()
                    line = f"{' '.join(labels)} {clean_text}\n"
                    f.write(line)
            except Exception as e:
                print(f"Skipping row {start + i} due to error: {e}")


In [14]:
# === Full Script to Check FastText Training Data ===

import os
from collections import Counter

# Set the correct path to your FastText training file
FILE_PATH = "fasttext_training_data.txt"  # or "fasttext_training_data.txt"

# Step 1: Check if file exists
if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(f"File not found: {FILE_PATH}")

# Step 2: Initialize counters
total_lines = 0
label_counter = Counter()

# Step 3: Read and analyze the file
with open(FILE_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        if not line.strip():
            continue  # skip empty lines
        total_lines += 1
        words = line.strip().split()
        labels = [w for w in words if w.startswith("__label__")]
        label_counter.update(labels)

        # Print a sample every 5 million lines
        if i % 5_000_000 == 0:
            print(f"Processed {i:,} lines. Sample:")
            print(line.strip())

# Step 4: Summary
print(f"\n✅ Finished reading file: {FILE_PATH}")
print(f"📊 Total lines (samples): {total_lines:,}")
print(f"🏷️  Unique labels found: {len(label_counter)}")
print("\nTop 10 most common labels:")
for label, count in label_counter.most_common(10):
    print(f"{label:<35} {count:,}")


Processed 5,000,000 lines. Sample:
__label__refund_and_returns one star stopped working month wiring issues suspect return period buy

✅ Finished reading file: fasttext_training_data.txt
📊 Total lines (samples): 7,609,969
🏷️  Unique labels found: 11

Top 10 most common labels:
__label__refund_and_returns         2,960,728
__label__product_quality            1,944,399
__label__long_wait_times            1,888,973
__label__delivery_delay             904,224
__label__fake_counterfeit           379,165
__label__packaging_handling         163,506
__label__technical_support          93,515
__label__order_issue                85,024
__label__rare_issues                44,491
__label__minor_issues               34,528


In [11]:
import vaex

# Open your dataset
df = vaex.open("processed_data/")  # adjust filename as needed

# Define issue columns
issue_cols = [
    "refund_and_returns", "product_quality", "long_wait_times", "delivery_delay",
    "fake_counterfeit", "packaging_handling", "technical_support", "order_issue",
    "rare_issues", "minor_issues", "warranty_guarantee"
]

# Sum columns manually row-wise
df["label_count"] = (
    df["refund_and_returns"] + df["product_quality"] + df["long_wait_times"] +
    df["delivery_delay"] + df["fake_counterfeit"] + df["packaging_handling"] +
    df["technical_support"] + df["order_issue"] + df["rare_issues"] +
    df["minor_issues"] + df["warranty_guarantee"]
)

# Count multi-label and single-label rows
multi_label_count = df[df.label_count > 1].shape[0]
single_label_count = df[df.label_count == 1].shape[0]

print(f"Rows with multiple labels: {multi_label_count:,}")
print(f"Rows with only one label: {single_label_count:,}")


Rows with multiple labels: 798,269
Rows with only one label: 6,811,700


In [14]:
import vaex

# Load your Vaex-compatible Parquet data
df = vaex.open("D:/jn/Amazon_cs_proj/Issue_categor_proj_Phase1/final_phase_ready_vaex/*.parquet")

# Define the binary multi-label issue columns
issue_columns = [
    'refund_and_returns', 'delivery_delay', 'product_quality',
    'order_issue', 'packaging_handling', 'long_wait_times',
    'technical_support', 'warranty_guarantee', 'fake_counterfeit',
    'rare_issues', 'minor_issues'
]

# Define your text column
text_column = 'combined_review'

# Output FastText training file
output_file = "fasttext_training_data.txt"

# Use chunking to avoid memory overflow
chunk_size = 1_000_000
total_rows = len(df)

with open(output_file, "w", encoding="utf-8") as f:
    for start in range(0, total_rows, chunk_size):
        end = min(start + chunk_size, total_rows)
        chunk = df[start:end]

        texts = chunk[text_column].tolist()
        all_labels = [chunk[col].tolist() for col in issue_columns]

        for i in range(len(texts)):
            try:
                text = texts[i]
                if not isinstance(text, str) or not text.strip():
                    continue  # Skip blank or invalid text

                labels = [
                    f"__label__{issue_columns[j]}"
                    for j in range(len(issue_columns))
                    if all_labels[j][i] == 1
                ]

                if labels:
                    clean_text = text.replace("\n", " ").strip()
                    line = f"{' '.join(labels)} {clean_text}\n"
                    f.write(line)
            except Exception as e:
                print(f"⚠️ Skipping row {start + i} due to error: {e}")

print(f"✅ Done: Exported FastText multi-label data to {output_file}")


ArrowMemoryError: realloc of size 805306368 failed